In [1]:
// 1. импорты для сессий и спарк sql
import $ivy.`org.apache.spark::spark-core:3.5.6`
import $ivy.`org.apache.spark::spark-sql:3.5.6`
import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.{SparkSession}

// 2. стоп сессий
SparkSession.getActiveSession.foreach(_.stop())

// 3. Конфиг и создание сессий
val conf = new SparkConf()
  .setAppName("DataSet")
  .setMaster("local[*]")
  .set("spark.driver.memory","1g")
  .set("spark.log.level", "WARN")

val spark = SparkSession.builder().config(conf).getOrCreate()
val sc = spark.sparkContext
println(s"Spark version: ${spark.version}")

// 4. методы для преобразования скала коллекций в датафреймы 
import spark.implicits._


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/21 00:18:20 WARN Utils: Your hostname, ubuntusd resolves to a loopback address: 127.0.1.1; using 192.168.1.169 instead (on interface ens192)
25/12/21 00:18:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/21 00:18:20 INFO SparkContext: Running Spark version 3.5.6
25/12/21 00:18:20 INFO SparkContext: OS info Linux, 5.4.0-216-generic, amd64
25/12/21 00:18:20 INFO SparkContext: Java version 11.0.27
25/12/21 00:18:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


Spark version: 3.5.6


import $ivy.$
import $ivy.$
import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.{SparkSession}
conf: SparkConf = org.apache.spark.SparkConf@604961b8
spark: SparkSession = org.apache.spark.sql.SparkSession@2746891a
sc: SparkContext = org.apache.spark.SparkContext@32109c54
import spark.implicits._

In [2]:
import org.apache.spark.sql.types._
// 5. кейс классы для DataSet
object CaseClasses {
    // case класс для паркет файла
    case class YellowTaxi(
        VendorID: Option[Int],
        tpep_pickup_datetime: Option[java.sql.Timestamp],
        tpep_dropoff_datetime: Option[java.sql.Timestamp],
        passenger_count: Option[Int],
        trip_distance: Option[Double],
        RatecodeID: Option[Int],
        store_and_fwd_flag: Option[String],
        PULocationID: Option[Int],
        DOLocationID: Option[Int],
        payment_type: Option[Int],
        fare_amount: Option[Double],
        extra: Option[Double],
        mta_tax: Option[Double],
        tip_amount: Option[Double],
        tolls_amount: Option[Double],
        improvement_surcharge: Option[Double],
        total_amount: Option[Double] 
    )

    // case класс для csv файла
    case class Zone(
        locationID: Option[Int],
        borough: Option[String],
        zone: Option[String],
        serviceZone: Option[String]
    )

    // case класс для итоговой витрины
    case class FinalTable(
        locationID: Option[Int],
        borough: Option[String],
        zone: Option[String],
        serviceZone: Option[String],
        count_trip: BigInt,
        avg_trip_distance: Double,
        min_trip_distance: Double,
        max_trip_distance: Double,
        stddev_trip_distance: Double
    )    
}

import org.apache.spark.sql.types._
defined object CaseClasses

In [ ]:
import CaseClasses.{Zone, YellowTaxi, FinalTable}
import org.apache.spark.sql.functions.{upper, trim, col, count, avg, min, max, stddev, broadcast}
import org.apache.spark.sql.{Encoders}

// 6. читаем источники в dataset'ы
val yellowTaxiDS = spark.read
  .parquet("yellow_taxi_jan_25_2018")
  .as[YellowTaxi]
// yellowTaxiDS.show(5)

val zoneDS = spark.read
  .option("header",true)
  .option("quote", "\"")
  .option("escape", "\"")
  .option("multiline", true)
  .option("ignoreLeadingWhiteSpace", true)
  .option("ignoreTrailingWhiteSpace", true)
  .schema(Encoders.product[Zone].schema) // схему берем из case класса
  .csv("taxi_zones.csv")
  .withColumn("serviceZone", upper(trim(col("serviceZone"))))
  .as[Zone]
// zoneDS.show(5)

// 7. описываем итоговую витрину, каталист использует броадкаст без подсказок для маленького набора zoneDS
val finalTableDS = yellowTaxiDS
    .select("PULocationID", "trip_distance")
    .groupBy("PULocationID")
    .agg(
        count("*").as("count_trip"),
        avg("trip_distance").as("avg_trip_distance"),
        min("trip_distance").as("min_trip_distance"),
        max("trip_distance").as("max_trip_distance"),
        stddev("trip_distance").as("stddev_trip_distance"))
    // способы подсказать выполнить броадкаст
    // .join(broadcast(zoneDS),col("PULocationID") === col("locationID"),"right")
    // .hint("broadcast")
    .join(zoneDS,col("PULocationID") === col("locationID"),"right")
    // .join(zoneDS,col("PULocationID") === col("locationID"),"right")
    .select(
      "locationID",
      "borough",
      "zone",
      "serviceZone",
      "count_trip",
      "avg_trip_distance",
      "min_trip_distance",
      "max_trip_distance",
      "stddev_trip_distance").as[FinalTable]

// finalTableDS.explain(true) // смоторим план, проверяем бродкаст   
// finalTableDS.show(5)

// 8. запись итоговой витрины в паркет файл
finalTableDS
  .orderBy("locationID")
  .write.mode("overwrite")
  .option("compression", "snappy")
  .partitionBy("locationID")
  .parquet("taxi_zone_statistics")

println(s"Final table count: ${finalTableDS.count()}")    

25/12/21 00:34:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: LocationID, Borough, Zone, service_zone
 Schema: locationID, borough, zone, serviceZone
Expected: serviceZone but found: service_zone
CSV file: file:///home/eugeny/taxi_zones.csv
25/12/21 00:34:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: LocationID, Borough, Zone, service_zone
 Schema: locationID, borough, zone, serviceZone
Expected: serviceZone but found: service_zone
CSV file: file:///home/eugeny/taxi_zones.csv
25/12/21 00:34:10 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: LocationID, Borough, Zone, service_zone
 Schema: locationID, borough, zone, serviceZone
Expected: serviceZone but found: service_zone
CSV file: file:///home/eugeny/taxi_zones.csv


Final table count: 265


import CaseClasses.{Zone, YellowTaxi, FinalTable}
import org.apache.spark.sql.functions.{upper, trim, col, count, avg, min, max, stddev, broadcast}
import org.apache.spark.sql.{Encoders}
yellowTaxiDS: org.apache.spark.sql.Dataset[YellowTaxi] = [VendorID: int, tpep_pickup_datetime: timestamp ... 15 more fields]
zoneDS: org.apache.spark.sql.Dataset[Zone] = [locationID: int, borough: string ... 2 more fields]
finalTableDS: org.apache.spark.sql.Dataset[FinalTable] = [locationID: int, borough: string ... 7 more fields]